# Spike Experimental Validation

Compare model predictions to experimentally measured viral titer fold changes
for 5 validation mutations.

In [ ]:
config_path = "config/config.yaml"
profile = None

In [ ]:
import warnings
import pickle

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import multidms

from notebooks._common import load_config, combine_replicate_muts

In [ ]:
cfg = load_config(config_path, profile)
spike = cfg["spike"]
output_dir = spike["output_dir"]
chosen_lasso_strength = spike["chosen_lasso_strength"]
times_seen_threshold = spike["times_seen_threshold"]
validation_mutations = spike.get("validation_mutations", ["D142L", "A419S", "A570D", "K854N", "T1027I"])

warnings.simplefilter("ignore")
%matplotlib inline
plt.rcParams.update({"legend.frameon": False, "font.size": 11})

In [ ]:
models = pickle.load(open(f"{output_dir}/full_models.pkl", "rb"))
print(f"Loaded {len(models)} models")

## Compute predicted phenotypic effects

In [ ]:
mut_df_replicates = combine_replicate_muts(
    {
        f"{fit.dataset_name}".split("-")[-1]: fit.model
        for fit in models.query(f"fusionreg == {chosen_lasso_strength}").itertuples()
    },
    predicted_func_scores=True,
    phenotype_as_effect=True,
    how="inner",
    times_seen_threshold=times_seen_threshold,
)

mut_df = (
    mut_df_replicates.assign(
        phenotypic_effect_Delta=2 ** mut_df_replicates["avg_predicted_func_score_Delta"],
        phenotypic_effect_Omicron_BA2=2 ** mut_df_replicates["avg_predicted_func_score_Omicron_BA2"],
        phenotypic_effect_Omicron_BA1=2 ** mut_df_replicates["avg_predicted_func_score_Omicron_BA1"],
    )
    .reset_index()
)
print(f"Computed phenotypic effects for {len(mut_df)} mutations")

## Viral titer comparison

In [ ]:
saveas = "validation_titer_fold_change"

row1 = ["titer", "titer", "."]
row2 = ["D142L", "A419S", "A570D"]
row3 = ["K854N", "T1027I", "legend"]
empty_row = ["."] * 3

fig = plt.figure(figsize=[6.4, 7])
axs = fig.subplot_mosaic(
    [row1, empty_row, row2, row3],
    height_ratios=[1, 0.39, 0.7, 0.7],
    empty_sentinel=".",
    gridspec_kw={"wspace": 0.20, "hspace": 0.4},
)

# TITERS
titers_df = pd.read_csv("data/viral_titers.csv")
titers_df.rename(columns={"RLUperuL": "titer", "background": "homolog"}, inplace=True)
titers_df["replicate"] = titers_df["virus"].apply(lambda x: x[-1])
titers_df["mutation"] = titers_df["virus"].str.extract(r"_(\S+)_")
titers_df["mutation"].fillna("unmutated", inplace=True)
titers_df["mutation"].replace("142L", "D142L", inplace=True)

homologs = ["Delta", "BA.1", "BA.2"]
xticklabels = ["unmutated"] + validation_mutations
for i, homolog in enumerate(homologs):
    data = titers_df[titers_df["homolog"] == homolog]
    sns.stripplot(
        x="mutation", y="titer", data=data, ax=axs["titer"],
        order=xticklabels, s=10, alpha=0.5,
        hue="homolog", hue_order=homologs,
    )
    sns.boxplot(
        x="mutation", y="titer", data=data, ax=axs["titer"],
        order=xticklabels, showfliers=False, showbox=False, showcaps=False,
        medianprops={"visible": False}, whiskerprops={"visible": False},
    )
handles, labels = axs["titer"].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
axs["titer"].legend(by_label.values(), by_label.keys(), bbox_to_anchor=[1, 0.5])
axs["titer"].set_yscale("log")
axs["titer"].set_yticks([1e2, 1e4, 1e6])
axs["titer"].set_xticklabels(axs["titer"].get_xticklabels(), rotation=90)
axs["titer"].set_ylabel(r"viral titer (RLU/$\mu$L)")
axs["titer"].set_xlabel("")
axs["titer"].grid()
sns.despine(ax=axs["titer"])

# FOLD CHANGE
val_df = pd.read_csv("data/spike_validation_data.csv")
val_dict = {k: [] for k in ["mutation", "fold_change", "homolog", "replicate", "predicted_beta"]}
for i, row in val_df.iterrows():
    for mutation in validation_mutations:
        homolog = row["background"].replace(".", "")
        homolog = "Omicron_" + homolog if "BA" in homolog else homolog
        matching = mut_df[mut_df["mutation"] == mutation][f"phenotypic_effect_{homolog}"]
        if len(matching) == 0:
            continue
        val_dict["mutation"].append(mutation)
        val_dict["fold_change"].append(row[mutation])
        val_dict["homolog"].append(homolog)
        val_dict["replicate"].append(row["replicate"])
        val_dict["predicted_beta"].append(float(matching.values[0]))

val_df = pd.DataFrame(val_dict)
val_df["site"] = val_df["mutation"].apply(lambda x: int(x[1:-1]))
val_df["homolog"].replace({"Omicron_BA1": "BA.1", "Omicron_BA2": "BA.2"}, inplace=True)
val_df.sort_values("site", inplace=True)

for mutation in validation_mutations:
    data = val_df[val_df["mutation"] == mutation]
    iter_ax = axs[mutation]
    sns.scatterplot(
        x="predicted_beta", y="fold_change", data=data,
        hue="homolog", ax=iter_ax, s=100, alpha=0.7, hue_order=homologs,
    )
    iter_ax.set(title=mutation, xlabel="", ylabel="", yscale="log", ylim=[1e-5, 2], yticks=[1, 1e-2, 1e-4])
    iter_ax.set_xscale("log", base=2)
    iter_ax.set_xlim([0.1, 1.3])
    iter_ax.set_xticks([2 ** -3, 2 ** -2, 2 ** -1, 2 ** 0])
    iter_ax.grid()
    if iter_ax.get_legend() is not None:
        iter_ax.get_legend().remove()
    sns.despine(ax=iter_ax)
    if mutation in ["D142L", "A419S"]:
        iter_ax.tick_params(axis="x", labelbottom=False)
    if mutation not in ["D142L", "K854N"]:
        iter_ax.tick_params(axis="y", labelleft=False)

fig.text(0.5, 0.02, "predicted enrichment ratio in DMS experiment\n(mutant : unmutated)", ha="center")
fig.text(0.000, 0.31, "fold change in viral titer\n (mutant : unmutated)", va="center", rotation="vertical")
axs["legend"].set_axis_off()
axs["titer"].text(-0.05, 1.15, "A", ha="right", va="center", size=15, weight="bold", transform=axs["titer"].transAxes)
axs["D142L"].text(-0.1, 1.25, "B", ha="right", va="center", size=15, weight="bold", transform=axs["D142L"].transAxes)

fig.savefig(f"{output_dir}/{saveas}.pdf", bbox_inches="tight")
plt.show()